In [7]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from lifelines.utils import concordance_index
import numpy as np
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

In [8]:
# --- CONFIGURATION ---
INPUT_DIR = "processed_data/"
MODEL_DIR = "saved_models/"
ENSEMBLE_SIZE = 3           # Train 3 models for the deep ensemble
EPOCHS = 100                # Max epochs per model
PATIENCE = 15               # Early stopping patience
LEARNING_RATE = 1e-4
RANKING_WEIGHT = 0.5        # Lambda for the pairwise ranking loss
ESM_DIM = 1280              # Use 1280 if using 650M model, 320 if using 8M model

In [9]:
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [11]:
# --- 1. NEURAL NETWORK ARCHITECTURE ---
class DTA_Model(nn.Module):
    def __init__(self, node_features=29, esm_dim=ESM_DIM, hidden_dim=256):
        super(DTA_Model, self).__init__()
        
        # Drug Graph Encoder (GCN)
        self.conv1 = GCNConv(node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim * 2)
        self.conv3 = GCNConv(hidden_dim * 2, hidden_dim)
        
        # Protein Feature Projector
        self.prot_proj = nn.Sequential(
            nn.Linear(esm_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, hidden_dim)
        )
        
        # MLP Output Head (Predicts Affinity)
        self.fc1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, int(hidden_dim/2))
        self.out = nn.Linear(int(hidden_dim/2), 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        target_emb = data.target_emb
        
        # 1. Process Drug Graph
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        # Global pooling collapses the graph into a single 1D vector per batch
        drug_rep = global_mean_pool(x, batch) 
        
        # 2. Process Protein Embedding
        prot_rep = self.prot_proj(target_emb)
        
        # 3. Concatenate and Predict
        cat_rep = torch.cat([drug_rep, prot_rep], dim=1)
        
        h = F.relu(self.fc1(cat_rep))
        h = self.dropout(h)
        h = F.relu(self.fc2(h))
        pred = self.out(h)
        
        return pred.squeeze(-1)

In [12]:
# --- 2. MULTI-OBJECTIVE GROUP DRO LOSS ---
def compute_group_loss(preds, targets, mask, ranking_weight):
    """Computes MSE + Pairwise Ranking loss for a specific subset."""
    if mask.sum() < 2:
        # If less than 2 samples in this group, we can't do ranking. Fallback to MSE.
        if mask.sum() == 1: return F.mse_loss(preds[mask], targets[mask])
        return torch.tensor(0.0, device=preds.device)

    group_preds = preds[mask]
    group_targets = targets[mask]

    # Objective 1: Regression (MSE)
    mse_loss = F.mse_loss(group_preds, group_targets)

    # Objective 2: Pairwise Ranking (Margin Loss for Concordance Index)
    # Create matrices of pairwise differences
    diff_preds = group_preds.unsqueeze(0) - group_preds.unsqueeze(1)
    diff_targets = group_targets.unsqueeze(0) - group_targets.unsqueeze(1)

    # We only care about pairs where True Target i > True Target j
    valid_pairs = diff_targets > 0
    if valid_pairs.sum() > 0:
        # We want predicted diff to be > margin (e.g., 0.1)
        margin = 0.1
        # max(0, margin - (pred_i - pred_j))
        rank_loss = torch.clamp(margin - diff_preds[valid_pairs], min=0.0).mean()
    else:
        rank_loss = torch.tensor(0.0, device=preds.device)

    return mse_loss + (ranking_weight * rank_loss)

def compute_total_loss(preds, data, ranking_weight):
    """Group DRO: Computes loss for A, B, and C, and optimizes for the worst group."""
    loss_A = compute_group_loss(preds, data.y, data.in_A == 1, ranking_weight)
    loss_B = compute_group_loss(preds, data.y, data.in_B == 1, ranking_weight)
    loss_C = compute_group_loss(preds, data.y, data.in_C == 1, ranking_weight)
    
    # Standard Group DRO: Heavily penalize based on the worst-performing subset
    active_losses = [l for l in [loss_A, loss_B, loss_C] if l > 0]
    
    if not active_losses:
        return F.mse_loss(preds, data.y) # Fallback
        
    return torch.max(torch.stack(active_losses))

In [13]:
# --- 3. EVALUATION SCRIPT ---
def evaluate_model(model, val_loader):
    model.eval()
    all_preds, all_targets = [], []
    all_A, all_B, all_C = [], [], []
    
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            preds = model(data)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(data.y.cpu().numpy())
            all_A.extend(data.in_A.cpu().numpy())
            all_B.extend(data.in_B.cpu().numpy())
            all_C.extend(data.in_C.cpu().numpy())
            
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    mask_A = np.array(all_A) == 1
    mask_B = np.array(all_B) == 1
    mask_C = np.array(all_C) == 1
    
    # Calculate RMSE
    rmse = np.sqrt(np.mean((all_preds - all_targets)**2))
    
    # Calculate Concordance Indices
    def safe_ci(y_true, y_pred, mask):
        if mask.sum() < 2: return 0.5 # Neutral baseline if missing
        return concordance_index(y_true[mask], y_pred[mask])
        
    ci_a = safe_ci(all_targets, all_preds, mask_A)
    ci_b = safe_ci(all_targets, all_preds, mask_B)
    ci_c = safe_ci(all_targets, all_preds, mask_C)
    
    # Competition Metric
    sr_ci = (0.30 * ci_a) + (0.35 * ci_b) + (0.35 * ci_c)
    
    return rmse, sr_ci, ci_a, ci_b, ci_c

In [14]:
# --- 4. MAIN ENSEMBLE TRAINING LOOP ---
print("Loading data binaries from disk...")
train_data = torch.load(os.path.join(INPUT_DIR, "train_data.pt"))
val_data = torch.load(os.path.join(INPUT_DIR, "val_data.pt"))

from torch.utils.data import WeightedRandomSampler
from torch_geometric.loader import DataLoader

# Re-initialize loaders
train_weights = [d.sample_weight.item() for d in train_data]
sampler = WeightedRandomSampler(weights=train_weights, num_samples=len(train_weights), replacement=True)
train_loader = DataLoader(train_data, batch_size=128, sampler=sampler, drop_last=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False)

print(f"\n--- Starting Ensemble Training ({ENSEMBLE_SIZE} Models) ---")

for ensemble_idx in range(1, ENSEMBLE_SIZE + 1):
    print(f"\n=== Training Model {ensemble_idx}/{ENSEMBLE_SIZE} ===")
    
    model = DTA_Model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    
    best_sr_ci = -float('inf')
    best_rmse = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
            batch = batch.to(device)
            optimizer.zero_grad()
            
            preds = model(batch)
            loss = compute_total_loss(preds, batch, RANKING_WEIGHT)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            
        avg_loss = total_loss / len(train_data)
        
        # Validate
        rmse, sr_ci, ci_a, ci_b, ci_c = evaluate_model(model, val_loader)
        
        print(f"Ep {epoch:03d} | Train Loss: {avg_loss:.4f} | Val RMSE: {rmse:.4f} | SR-CI: {sr_ci:.4f} (A:{ci_a:.2f}, B:{ci_b:.2f}, C:{ci_c:.2f})")
        
        # Model Selection (Tie-breaker on RMSE)
        if sr_ci > best_sr_ci or (np.isclose(sr_ci, best_sr_ci) and rmse < best_rmse):
            best_sr_ci = sr_ci
            best_rmse = rmse
            epochs_no_improve = 0
            
            checkpoint_path = os.path.join(MODEL_DIR, f"dta_ensemble_model_{ensemble_idx}.pt")
            torch.save(model.state_dict(), checkpoint_path)
            print(f" -> Checkpoint saved (New Best SR-CI)!")
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered! Best SR-CI for Model {ensemble_idx}: {best_sr_ci:.4f}")
            break

print("\n--- Phase 3 Complete ---")

Loading data binaries from disk...

--- Starting Ensemble Training (3 Models) ---

=== Training Model 1/3 ===


Ep 001 | Train Loss: 7.4464 | Val RMSE: 0.8187 | SR-CI: 0.6681 (A:0.67, B:0.66, C:0.67)
 -> Checkpoint saved (New Best SR-CI)!


Ep 002 | Train Loss: 0.9365 | Val RMSE: 0.8186 | SR-CI: 0.7025 (A:0.71, B:0.70, C:0.70)
 -> Checkpoint saved (New Best SR-CI)!


Ep 003 | Train Loss: 0.8918 | Val RMSE: 0.7939 | SR-CI: 0.7123 (A:0.72, B:0.71, C:0.71)
 -> Checkpoint saved (New Best SR-CI)!


Ep 004 | Train Loss: 0.8719 | Val RMSE: 0.7836 | SR-CI: 0.7271 (A:0.73, B:0.73, C:0.73)
 -> Checkpoint saved (New Best SR-CI)!


...

Ep 081 | Train Loss: 0.5939 | Val RMSE: 1.2751 | SR-CI: 0.8079 (A:0.81, B:0.80, C:0.81)


Ep 082 | Train Loss: 0.5891 | Val RMSE: 1.0769 | SR-CI: 0.8053 (A:0.81, B:0.80, C:0.81)


Ep 083 | Train Loss: 0.6136 | Val RMSE: 1.1753 | SR-CI: 0.8038 (A:0.81, B:0.80, C:0.81)


Ep 084 | Train Loss: 0.6120 | Val RMSE: 1.2407 | SR-CI: 0.8067 (A:0.81, B:0.80, C:0.81)


Ep 085 | Train Loss: 0.5838 | Val RMSE: 1.2904 | SR-CI: 0.8056 (A:0.81, B:0.80, C:0.81)
Early stopping triggered! Best SR-CI for Model 3: 0.8083

--- Phase 3 Complete ---
